# California Housing Price Prediction: OLS vs Ridge vs Lasso

**Objective:** Predict the **median house value** of California districts. The pipeline covers missing-value imputation, feature engineering (ratio and interaction features), multicollinearity diagnosis (correlation heatmap and VIF), scaling and encoding, and a comparison of **OLS, Ridge (L2) and Lasso (L1)** regression using **RMSE, MAE and R²**. Hyperparameters of Ridge and Lasso are tuned with `GridSearchCV`.

**Pipeline:** Load → EDA → Feature engineering → Multicollinearity check → Split → Impute / Scale / Encode → Train (GridSearchCV) → Evaluate → Residual analysis

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_theme(style="whitegrid", context="notebook")
RANDOM_STATE = 42

## 2. Load the Dataset

In [ ]:
URL = "https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv"

try:
    housing = pd.read_csv(URL)
    print("Loaded dataset from URL")
except Exception as e:
    print("Could not load from URL:", e)
    print("Trying a local copy named 'housing.csv' placed next to this notebook...")
    housing = pd.read_csv("housing.csv")

print("Shape:", housing.shape)
housing.head()

## 3. Exploratory Data Analysis and Missing Values

In [ ]:
housing.info()
print("\nMissing values per column:")
print(housing.isnull().sum()[housing.isnull().sum() > 0])
print("\nocean_proximity categories:\n", housing["ocean_proximity"].value_counts())
housing.describe().T.round(2)

In [ ]:
# Distributions of all numeric columns
housing.hist(bins=40, figsize=(14, 9), color="steelblue", edgecolor="white")
plt.suptitle("Feature Distributions", y=1.01, fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Geographic view: price is strongly tied to location (coast) and population density
plt.figure(figsize=(8, 6))
sc = plt.scatter(housing["longitude"], housing["latitude"], c=housing["median_house_value"],
                 s=housing["population"] / 100, cmap="viridis", alpha=0.4)
plt.colorbar(sc, label="Median house value ($)")
plt.xlabel("Longitude"); plt.ylabel("Latitude")
plt.title("California Districts: Price by Location (bubble size = population)")
plt.show()

**Notes**
- Only `total_bedrooms` has missing values. These are imputed with the **median** inside the modelling pipeline (fitted on training data only, to avoid leakage).
- `median_house_value` is **capped at $500,001**, so a cluster of points sits at the ceiling. This limits how well any linear model can do at the top end.
- `ocean_proximity` is categorical and will be one-hot encoded.

## 4. Feature Engineering (Ratios and Interaction Features)

In [ ]:
def add_features(d):
    d = d.copy()
    # Ratio features: raw totals depend on district size, ratios describe the typical household
    d["rooms_per_household"]      = d["total_rooms"] / d["households"]
    d["bedrooms_per_room"]        = d["total_bedrooms"] / d["total_rooms"]
    d["population_per_household"] = d["population"] / d["households"]
    # Interaction features. Variables are mean-centred first: multiplying raw, uncentred columns creates
    # a product that is almost a copy of its parents (huge artificial multicollinearity).
    # Only feature means are used (no target information).
    c = lambda s: s - s.mean()
    d["income_x_rooms"] = c(d["median_income"]) * c(d["rooms_per_household"])
    d["income_x_age"]   = c(d["median_income"]) * c(d["housing_median_age"])
    d["lat_x_lon"]      = c(d["latitude"]) * c(d["longitude"])
    return d

df = add_features(housing)
df[["rooms_per_household", "bedrooms_per_room", "population_per_household",
    "income_x_rooms", "income_x_age", "lat_x_lon"]].describe().T.round(3)

## 5. Correlation Heatmap and Multicollinearity Check

In [ ]:
num_cols = df.select_dtypes(include="number").columns
corr = df[num_cols].corr()

plt.figure(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            square=True, linewidths=0.5, annot_kws={"size": 8})
plt.title("Correlation Heatmap (raw + engineered features)")
plt.show()

print("Correlation with target (median_house_value):")
corr["median_house_value"].drop("median_house_value").sort_values(ascending=False).round(3)

**Variance Inflation Factor (VIF)** quantifies multicollinearity: `VIF = 1 / (1 - R²)`, where R² comes from regressing one feature on all the others. A common rule of thumb: VIF above 5 is moderate and above 10 is severe.

In [ ]:
def compute_vif(frame):
    X = frame.copy()
    X = X.fillna(X.median())
    X = (X - X.mean()) / X.std()
    out = {}
    for col in X.columns:
        others = X.drop(columns=col)
        r2 = LinearRegression().fit(others, X[col]).score(others, X[col])
        out[col] = 1.0 / (1.0 - r2) if r2 < 1 else np.inf
    return pd.Series(out).sort_values(ascending=False)

raw_features = ["longitude", "latitude", "housing_median_age", "total_rooms",
                "total_bedrooms", "population", "households", "median_income"]
vif_raw = compute_vif(housing[raw_features])

# Handle multicollinearity: replace the raw count columns (which move together with district size)
# by the per-household ratios. Interaction features are kept for the regularised models to judge.
count_cols = ["total_rooms", "total_bedrooms", "population", "households"]
vif_eng = compute_vif(df.drop(columns=count_cols + ["median_house_value", "ocean_proximity"]))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, vif, title in zip(axes, [vif_raw, vif_eng],
                          ["VIF: raw features", "VIF: after replacing raw counts with ratios"]):
    vif.sort_values().plot(kind="barh", ax=ax, color="steelblue")
    ax.axvline(5, color="orange", ls="--", label="VIF = 5")
    ax.axvline(10, color="red", ls="--", label="VIF = 10")
    ax.set_title(title); ax.set_xlabel("VIF"); ax.legend()
plt.tight_layout()
plt.show()

print("Raw features VIF:\n", vif_raw.round(2).to_string())
print("\nAfter handling (ratios replace counts):\n", vif_eng.round(2).to_string())

**How multicollinearity is handled here**
1. **Feature engineering:** the four raw count columns are strongly correlated with each other (they all scale with district size), so they are replaced by per-household ratios.
2. **Centring:** interaction terms are built from mean-centred variables, which removes the artificial correlation between a product and its parents.
3. **Regularisation:** latitude and longitude stay correlated by nature of geography. Ridge shrinks correlated coefficients together and Lasso can drop redundant ones, which is exactly what the model comparison below tests.

## 6. Train/Test Split and Preprocessing Pipeline

In [ ]:
model_df = df.drop(columns=count_cols)
X = model_df.drop(columns="median_house_value")
y = model_df["median_house_value"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

# Model the target in units of $100k so the coordinate-descent solver behind Lasso converges cleanly.
# Predictions are converted back to dollars before computing metrics.
Y_SCALE = 100_000
y_train_s = y_train / Y_SCALE

num_features = X.select_dtypes(include="number").columns.tolist()
cat_features = ["ocean_proximity"]

# Numeric: impute (median) then standardise.  Categorical: one-hot encode.
# Everything is fitted on the training folds only, so there is no data leakage.
preprocessor = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                      ("scaler", StandardScaler())]), num_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
])
print("Numeric features:", num_features)

## 7. Train Models: OLS, Ridge and Lasso (GridSearchCV)

In [ ]:
ols = Pipeline([("prep", preprocessor), ("model", LinearRegression())])

ridge_grid = GridSearchCV(
    Pipeline([("prep", preprocessor), ("model", Ridge(random_state=RANDOM_STATE))]),
    param_grid={"model__alpha": np.logspace(-3, 3, 13)},
    cv=5, scoring="neg_root_mean_squared_error", n_jobs=-1,
)

lasso_grid = GridSearchCV(
    Pipeline([("prep", preprocessor), ("model", Lasso(max_iter=50000, random_state=RANDOM_STATE))]),
    param_grid={"model__alpha": np.logspace(-5, -1, 13)},
    cv=5, scoring="neg_root_mean_squared_error", n_jobs=-1,
)

ols.fit(X_train, y_train_s)
ridge_grid.fit(X_train, y_train_s)
lasso_grid.fit(X_train, y_train_s)

print(f"Best Ridge alpha: {ridge_grid.best_params_['model__alpha']:.4g}  (CV RMSE: ${-ridge_grid.best_score_ * Y_SCALE:,.0f})")
print(f"Best Lasso alpha: {lasso_grid.best_params_['model__alpha']:.4g}  (CV RMSE: ${-lasso_grid.best_score_ * Y_SCALE:,.0f})")

fitted = {"OLS": ols, "Ridge (L2)": ridge_grid.best_estimator_, "Lasso (L1)": lasso_grid.best_estimator_}

In [ ]:
# How the CV error changes with regularisation strength
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, grid, name in zip(axes, [ridge_grid, lasso_grid], ["Ridge", "Lasso"]):
    alphas = grid.cv_results_["param_model__alpha"].astype(float)
    rmse = -grid.cv_results_["mean_test_score"] * Y_SCALE
    ax.semilogx(alphas, rmse, marker="o")
    ax.axvline(grid.best_params_["model__alpha"], color="red", ls="--", label="best alpha")
    ax.set_title(f"{name}: 5-fold CV RMSE vs alpha")
    ax.set_xlabel("alpha (log scale)"); ax.set_ylabel("CV RMSE ($)"); ax.legend()
plt.tight_layout()
plt.show()

## 8. Evaluation: RMSE, MAE and R²

In [ ]:
rows, preds = [], {}
for name, model in fitted.items():
    y_pred = model.predict(X_test) * Y_SCALE   # back to dollars
    preds[name] = y_pred
    rows.append({
        "Model": name,
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "MAE":  mean_absolute_error(y_test, y_pred),
        "R²":   r2_score(y_test, y_pred),
    })

results = pd.DataFrame(rows).set_index("Model")
display(results.style.format({"RMSE": "{:,.0f}", "MAE": "{:,.0f}", "R²": "{:.4f}"})
        .highlight_min(subset=["RMSE", "MAE"], color="#c8e6c9")
        .highlight_max(subset=["R²"], color="#c8e6c9"))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, metric in zip(axes, ["RMSE", "MAE", "R²"]):
    results[metric].plot(kind="bar", ax=ax, color=sns.color_palette("Set2", 3), rot=0)
    ax.set_title(metric)
    ax.bar_label(ax.containers[0], fmt="%.3f" if metric == "R²" else "%.0f", padding=2, fontsize=9)
    ax.set_xlabel("")
plt.suptitle("Model Comparison on Test Set", y=1.03)
plt.tight_layout()
plt.show()

## 9. Coefficients: Effect of L1 and L2 Regularisation

In [ ]:
feature_names = fitted["OLS"].named_steps["prep"].get_feature_names_out()
feature_names = [n.split("__")[-1] for n in feature_names]

# Coefficients converted to dollars: change in predicted price for a 1-std change in the feature
coefs = pd.DataFrame({name: m.named_steps["model"].coef_ * Y_SCALE for name, m in fitted.items()},
                     index=feature_names)

coefs.sort_values("OLS").plot(kind="barh", figsize=(10, 7), width=0.8)
plt.axvline(0, color="k", lw=0.8)
plt.title("Coefficients: OLS vs Ridge vs Lasso")
plt.xlabel("Effect on predicted price ($ per 1-std change in feature)")
plt.tight_layout()
plt.show()

zeroed = coefs.index[coefs["Lasso (L1)"].abs() < 1e-8].tolist()
print("Features eliminated by Lasso:", zeroed if zeroed else "none (all features retained at the chosen alpha)")
coefs.round(0)

## 10. Predicted vs Actual

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8), sharex=True, sharey=True)
lims = [0, max(y_test.max(), max(p.max() for p in preds.values()))]
for ax, (name, y_pred) in zip(axes, preds.items()):
    ax.scatter(y_test, y_pred, alpha=0.25, s=10, color="steelblue")
    ax.plot(lims, lims, "r--", lw=1.5, label="Perfect prediction")
    ax.set_title(f"{name}\nR² = {r2_score(y_test, y_pred):.3f}")
    ax.set_xlabel("Actual median house value ($)")
    ax.legend(loc="upper left")
axes[0].set_ylabel("Predicted median house value ($)")
plt.tight_layout()
plt.show()

## 11. Residual Analysis

In [ ]:
residuals = {name: y_test.values - y_pred for name, y_pred in preds.items()}

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))

# Residual distribution curves (should be roughly centred on 0 and symmetric)
for name, r in residuals.items():
    sns.kdeplot(r, ax=axes[0], label=f"{name} (mean={r.mean():,.0f}, std={r.std():,.0f})", fill=True, alpha=0.2)
axes[0].axvline(0, color="k", lw=1)
axes[0].set_title("Residual Distribution (Actual − Predicted)")
axes[0].set_xlabel("Residual ($)")
axes[0].legend(fontsize=8)

# Residuals vs predicted (a funnel shape would indicate heteroscedasticity)
best_name = results["RMSE"].idxmin()
axes[1].scatter(preds[best_name], residuals[best_name], alpha=0.25, s=10, color="darkorange")
axes[1].axhline(0, color="k", lw=1)
axes[1].set_title(f"Residuals vs Predicted ({best_name})")
axes[1].set_xlabel("Predicted value ($)"); axes[1].set_ylabel("Residual ($)")

plt.tight_layout()
plt.show()

In [ ]:
# Histogram of residuals for each model
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, (name, r) in zip(axes, residuals.items()):
    sns.histplot(r, bins=60, kde=True, ax=ax, color="steelblue")
    ax.axvline(0, color="red", ls="--")
    ax.set_title(name); ax.set_xlabel("Residual ($)")
plt.tight_layout()
plt.show()

## 12. Summary

In [ ]:
best = results["RMSE"].idxmin()
print(f"Best model by RMSE: {best}")
print(f"  RMSE = ${results.loc[best, 'RMSE']:,.0f}   MAE = ${results.loc[best, 'MAE']:,.0f}   R² = {results.loc[best, 'R²']:.4f}")
print(f"\nRidge best alpha: {ridge_grid.best_params_['model__alpha']:.4g}")
print(f"Lasso best alpha: {lasso_grid.best_params_['model__alpha']:.4g}")
print("Lasso eliminated:", zeroed if zeroed else "no features")

## Key Takeaways and Discussion

| Aspect | OLS | Ridge (L2) | Lasso (L1) |
|---|---|---|---|
| Penalty | None | Sum of squared coefficients | Sum of absolute coefficients |
| Effect on correlated features | Unstable, inflated coefficients | Shrinks them smoothly together | Keeps one, may drop others |
| Feature selection | No | No (shrinks, never exactly zero) | Yes (can set coefficients to 0) |
| Tuned hyperparameter | None | `alpha` | `alpha` |

- **Multicollinearity:** the raw count features show the highest VIFs. Replacing them with per-household ratios (and centring the interaction terms) reduces this, and regularisation handles the correlation that remains, mainly between latitude and longitude.
- **Why the three models can score similarly:** with about 16,000 training rows and only a handful of features, OLS has little variance to reduce, so regularisation gives modest gains in accuracy. Its main value here is stable, interpretable coefficients.
- **Limitations:** `median_house_value` is capped at $500,001, the relationship with location is non-linear, and residuals typically fan out for expensive districts (check the residual plots above). Tree-based models (Random Forest, Gradient Boosting) or spatial features would likely improve R² further, which is a natural next step.
- All preprocessing (imputation, scaling, encoding) is inside the pipeline and fitted on training folds only, so there is no data leakage.